# COMP4318/5318 Assignment 2: Image Classification

## Gaussian Naive Bayes - algorithm of choice

### Group number: Assignment 163 
### SID1: 530210074  
### SID2: 540868894  
### SID3: 550220550  
### SID4: 550574604

## Setup and dependencies

This notebook uses NumPy, pandas, Matplotlib, and scikit-learn. All random splitting uses the shared integer seed `5318`.

In [1]:
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)

SEED = 5318
np.random.seed(SEED)
CLASS_NAMES = {
    0: 'Adipose', 1: 'Background', 2: 'Debris', 3: 'Lymphocytes',
    4: 'Mucus', 5: 'Smooth muscle', 6: 'Normal colon mucosa',
    7: 'Cancer-associated stroma', 8: 'Colorectal adenocarcinoma epithelium'
}


## 1. Data loading, exploration, and preprocessing

The Canvas training/test split is retained. A fixed stratified 80/20 split within the original training data is used only for development and hyperparameter selection; the original test set is reserved for the final frozen model.

In [2]:
relative = Path('Canvas Sources') / 'Assignment2Data-1' / 'Assignment2Data'
data_candidates = [base / part for base in (Path.cwd(), *Path.cwd().parents)
                   for part in (relative, Path('Assignment2Data'))]
DATA_DIR = next((p for p in data_candidates if (p / 'X_train.npy').is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Canvas .npy files not found relative to this notebook.')

X_train = np.load(DATA_DIR / 'X_train.npy', mmap_mode='r', allow_pickle=False)
y_train = np.load(DATA_DIR / 'y_train.npy', allow_pickle=False).reshape(-1).astype(int)
X_test = np.load(DATA_DIR / 'X_test.npy', mmap_mode='r', allow_pickle=False)
y_test = np.load(DATA_DIR / 'y_test.npy', allow_pickle=False).reshape(-1).astype(int)
fit_idx, val_idx = train_test_split(np.arange(len(y_train)), test_size=0.20,
                                    stratify=y_train, random_state=SEED)
assert set(fit_idx).isdisjoint(set(val_idx))
assert X_train.ndim == 4 and X_train.shape[1:] == X_test.shape[1:]
print(f'Training images: {X_train.shape}; test images: {X_test.shape}')
print('Classes:', np.unique(y_train).tolist())
print(f'Fit samples: {len(fit_idx)}; validation samples: {len(val_idx)}')


Training images: (32000, 28, 28, 3); test images: (8000, 28, 28, 3)
Classes: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Fit samples: 25600; validation samples: 6400


### Preprocessing for GaussianNB

GaussianNB expects a two-dimensional continuous feature matrix. Each RGB image is flattened from 28 x 28 x 3 to 2,352 features and fixed pixel scaling maps 0-255 to 0-1. This fixed rescaling does not use validation or test statistics. The model estimates one Gaussian distribution per class and pixel feature from the fitting data.

In [3]:
pixel_divisor = 255.0 if float(np.max(X_train)) > 1.0 + 1e-6 else 1.0
X_fit = np.asarray(X_train[fit_idx], dtype=np.float32).reshape(len(fit_idx), -1) / pixel_divisor
X_val = np.asarray(X_train[val_idx], dtype=np.float32).reshape(len(val_idx), -1) / pixel_divisor
y_fit, y_val = y_train[fit_idx], y_train[val_idx]
assert np.isfinite(X_fit).all() and np.isfinite(X_val).all()
print(f'Pixel divisor: {pixel_divisor:.1f}')
print('Flattened fit/validation shapes:', X_fit.shape, X_val.shape)
pixel_feature_variances = X_fit.var(axis=0)
print(f'Fit pixel range: [{X_fit.min():.4f}, {X_fit.max():.4f}]')
print('Per-feature pixel variance: min={:.6f}, max={:.6f}, median={:.6f}'.format(
    pixel_feature_variances.min(), pixel_feature_variances.max(),
    np.median(pixel_feature_variances)
))


Pixel divisor: 255.0
Flattened fit/validation shapes: (25600, 2352) (6400, 2352)
Fit pixel range: [0.0000, 1.0000]
Per-feature pixel variance: min=0.015404, max=0.033659, median=0.015986


### Examples of preprocessed data

The following display shows the fixed [0, 1] rescaling used before flattening.

In [4]:
fig, axes = plt.subplots(3, 3, figsize=(7, 7))
for ax, index in zip(axes.ravel(), fit_idx[:9]):
    ax.imshow(np.asarray(X_train[index], dtype=np.float32) / pixel_divisor)
    ax.set_title(f'Class {y_train[index]}')
    ax.axis('off')
plt.tight_layout()
plt.show()


## 2. Algorithm design and setup

### Algorithm of choice from first six weeks of course: Gaussian Naive Bayes

GaussianNB models every flattened pixel feature as conditionally Gaussian within each class and applies Bayes' rule to combine class-conditional likelihoods. The naive assumption is conditional independence given the class. It is a compact, fast traditional classifier with no iterative optimisation, making it a useful contrast to the MLP and CNN. Its key limitation for colour microscopy images is that neighbouring pixels and RGB channels are strongly correlated, so the independence assumption is unlikely to hold exactly.

In [5]:
# Initial untuned instance. The test set is not used in this section.
initial_start = perf_counter()
initial_model = GaussianNB()
initial_model.fit(X_fit, y_fit)
initial_fit_seconds = perf_counter() - initial_start
initial_start = perf_counter()
initial_predictions = initial_model.predict(X_val)
initial_validation_seconds = perf_counter() - initial_start
print('Initial GaussianNB: fit={:.4f}s, validation={:.4f}s, accuracy={:.4f}, '
      'macro_f1={:.4f}, balanced_accuracy={:.4f}'.format(
          initial_fit_seconds, initial_validation_seconds,
          accuracy_score(y_val, initial_predictions),
          f1_score(y_val, initial_predictions, average='macro'),
          balanced_accuracy_score(y_val, initial_predictions)))


Initial GaussianNB: fit=0.2731s, validation=0.5977s, accuracy=0.4041, macro_f1=0.3749, balanced_accuracy=0.4188


### Fully connected neural network

To be integrated from the corresponding owner's notebook.

### Convolutional neural network

To be integrated from the corresponding owner's notebook.

## 3. Hyperparameter tuning

### Algorithm of choice from first six weeks of course

`var_smoothing` adds a small stabilising quantity to each estimated feature variance. A manual validation-set search is used because the shared fixed validation partition is the group standard. Macro-F1 is the selection metric, so every class contributes equally despite different class counts. This search cell is independent of the final-model cell.

**Search design expectation.** `var_smoothing` adds a multiple of the largest feature variance to each variance estimate. The observed per-pixel variances are approximately 0.015-0.034 after rescaling, so values much smaller than this are expected to form a performance plateau. Values comparable to or larger than the feature variances may wash out class-specific variance differences and reduce Macro-F1. This theoretical reasoning motivates the log-spaced range from 1e-12 to 1.

In [6]:
# Independent hyperparameter search. Do not use the test set here.
smoothing_values = [1e-12, 1e-10, 1e-8, 1e-6, 1e-4, 1e-2, 1e-1, 1.0]
search_records = []
for smoothing in smoothing_values:
    start = perf_counter()
    model = GaussianNB(var_smoothing=smoothing)
    model.fit(X_fit, y_fit)
    fit_seconds = perf_counter() - start
    start = perf_counter()
    predictions = model.predict(X_val)
    validation_seconds = perf_counter() - start
    search_records.append({
        'var_smoothing': smoothing,
        'validation_accuracy': accuracy_score(y_val, predictions),
        'validation_macro_f1': f1_score(y_val, predictions, average='macro'),
        'validation_balanced_accuracy': balanced_accuracy_score(y_val, predictions),
        'fit_seconds': fit_seconds,
        'validation_seconds': validation_seconds,
    })
search_results = pd.DataFrame(search_records)
search_display = search_results.copy()
search_display['var_smoothing'] = search_display['var_smoothing'].map(lambda value: f'{value:.0e}')
for column in [
    'validation_accuracy', 'validation_macro_f1', 'validation_balanced_accuracy',
    'fit_seconds', 'validation_seconds'
]:
    search_display[column] = search_display[column].map(lambda value: f'{value:.4f}')
display(search_display)


,var_smoothing,validation_accuracy,validation_macro_f1,validation_balanced_accuracy,fit_seconds,validation_seconds
0,1e-12,0.4041,0.3749,0.4188,0.2705,0.6418
1,1e-10,0.4041,0.3749,0.4188,0.2857,0.6210
2,1e-08,0.4041,0.3749,0.4188,0.2530,0.6369
3,1e-06,0.4041,0.3749,0.4188,0.3216,0.6447
4,1e-04,0.4039,0.3748,0.4186,0.2519,0.6364
5,1e-02,0.4028,0.3719,0.4172,0.2671,0.7005
6,1e-01,0.4027,0.3597,0.4161,0.2501,0.6195
7,1e+00,0.3345,0.2664,0.3494,0.2486,0.6940


In [7]:
# Visualise validation performance. Runtime points are recorded but not interpreted as a smoothing trend.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogx(
    search_results['var_smoothing'], search_results['validation_macro_f1'],
    marker='o', color='tab:blue'
)
axes[0].set_title('GaussianNB validation Macro-F1')
axes[0].set_xlabel('var_smoothing (log scale)')
axes[0].set_ylabel('Macro-F1')
axes[0].set_ylim(0, 0.40)
axes[0].grid(True, alpha=0.3)

axes[1].scatter(
    search_results['var_smoothing'], search_results['fit_seconds'],
    label='Fit time', color='tab:orange'
)
axes[1].scatter(
    search_results['var_smoothing'], search_results['validation_seconds'],
    label='Validation prediction time', color='tab:green'
)
axes[1].set_xscale('log')
axes[1].set_title('Observed runtime per search trial')
axes[1].set_xlabel('var_smoothing (log scale)')
axes[1].set_ylabel('Seconds')
axes[1].grid(True, alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()


### Search result interpretation

The smallest six smoothing values form a near-flat validation Macro-F1 plateau. This is consistent with their scale being negligible relative to the observed pixel variances. Larger smoothing values reduce Macro-F1, supporting the expectation that excessive variance smoothing removes useful class-specific information. The recorded runtime points are environment-dependent observations, not evidence that `var_smoothing` changes computational complexity.

Values from 1e-12 to 1e-6 tie at the displayed Macro-F1 value of 0.3749. `var_smoothing=1e-6` is selected only as a fixed tie-break choice: its added variance is negligible relative to the observed pixel-feature variance range, so it has no material effect on the fitted model. The final model below hard-codes this reproducible choice and does not depend on `search_results`.

### Fully connected neural network

To be integrated from the corresponding owner's notebook.

### Convolutional neural network

To be integrated from the corresponding owner's notebook.

## 4. Final models

### Algorithm of choice from first six weeks of course

The final GaussianNB is independently rebuilt with the frozen setting. It fits all Canvas training samples, then evaluates the original held-out test set. This cell remains runnable when the tuning cell is skipped.

In [8]:
# This final cell is deliberately independent of the search cell.
BEST_VAR_SMOOTHING = 1e-6
X_train_final = np.asarray(X_train, dtype=np.float32).reshape(len(y_train), -1) / pixel_divisor
X_test_final = np.asarray(X_test, dtype=np.float32).reshape(len(y_test), -1) / pixel_divisor

final_start = perf_counter()
final_model = GaussianNB(var_smoothing=BEST_VAR_SMOOTHING)
final_model.fit(X_train_final, y_train)
final_fit_seconds = perf_counter() - final_start
final_start = perf_counter()
final_predictions = final_model.predict(X_test_final)
final_predict_seconds = perf_counter() - final_start

print('Final GaussianNB parameters:', final_model.get_params())
print(f'Training time: {final_fit_seconds:.4f} seconds')
print(f'Test prediction time: {final_predict_seconds:.4f} seconds')
print(f'Test accuracy: {accuracy_score(y_test, final_predictions):.4f}')
print(f'Test macro F1: {f1_score(y_test, final_predictions, average="macro"):.4f}')
print(f'Test balanced accuracy: {balanced_accuracy_score(y_test, final_predictions):.4f}')
print()
print(classification_report(y_test, final_predictions, digits=4, zero_division=0))
cm = confusion_matrix(y_test, final_predictions, labels=np.arange(len(CLASS_NAMES)))
fig, ax = plt.subplots(figsize=(8, 6))
short_class_names = [
    'Adipose', 'Background', 'Debris', 'Lymphocytes', 'Mucus',
    'Smooth\nmuscle', 'Normal\nmucosa', 'Cancer\nstroma', 'Cancer\nepithelium'
]
ConfusionMatrixDisplay(cm, display_labels=short_class_names).plot(
    ax=ax, cmap='Blues', colorbar=False, values_format='d')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_title('GaussianNB test confusion matrix')
plt.tight_layout()
plt.show()


Final GaussianNB parameters: {'priors': None, 'var_smoothing': 1e-06}
Training time: 0.3691 seconds
Test prediction time: 0.7947 seconds
Test accuracy: 0.4155
Test macro F1: 0.3857
Test balanced accuracy: 0.4307

              precision    recall  f1-score   support

           0     0.6708    0.9381    0.7822       873
           1     0.6895    0.6445    0.6663       858
           2     0.1848    0.1026    0.1320       877
           3     0.4173    0.6214    0.4993       914
           4     0.3479    0.4749    0.4016       737
           5     0.4576    0.0756    0.1297      1072
           6     0.2702    0.2155    0.2398       682
           7     0.2731    0.6298    0.3810       813
           8     0.3871    0.1738    0.2399      1174

    accuracy                         0.4155      8000
   macro avg     0.4109    0.4307    0.3857      8000
weighted avg     0.4161    0.4155    0.3771      8000



### Fully connected neural network

To be integrated from the corresponding owner's notebook.

### Convolutional neural network

To be integrated from the corresponding owner's notebook.

## 5. AI Acknowledgement

ChatGPT (OpenAI) assisted with code scaffolding, debugging, and review. The group must review this implementation and update this acknowledgement to accurately reflect its final use before submission.

In [9]:
from pathlib import Path
import os, json
import numpy as np
from sklearn.metrics import *
_v2_dir = Path(os.environ['V2_RUN_DIR'])
np.save(_v2_dir / 'y_test.npy', np.asarray(y_test).reshape(-1))
np.save(_v2_dir / 'gaussian_nb_predictions.npy', np.asarray(final_predictions).reshape(-1))
search_results.to_csv(_v2_dir / 'search_results.csv', index=False)
(_v2_dir / 'parameter_inventory.json').write_text(json.dumps({'theta': int(final_model.theta_.size), 'variance': int(final_model.var_.size), 'class_prior': int(final_model.class_prior_.size), 'stored_numeric_total': int(final_model.theta_.size + final_model.var_.size + final_model.class_prior_.size)}, indent=2), encoding='utf-8')

print('V2 audit export complete:', _v2_dir)


V2 audit export complete: /workspace/repo/reproduction_v2/runs/20261027T084817Z_naive_bayes_c0d7f694
